In [20]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sys
import os
from sklearn.preprocessing import LabelEncoder


project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.nbaPlayerLogs import NBAGameLogs

pd.set_option('display.max_columns', None)

## Fetches Player Gamelogs

In [21]:
# df = NBAGameLogs(season='2024-25', season_type='Regular Season').fetch(skip_start_positions=True).build().get_df()
# df.head()

# Session by session — run this repeatedly until all games are done
logs = NBAGameLogs(season='2025-26', season_type='Playoffs')
logs.fetch(batch_size=50, start_position_delay=2.5, start_position_workers=5, checkpoint_path='tracking_checkpoint.csv')

Fetching data for 2025-26 Playoffs...
✓ Player base
✓ Player advanced
✓ Team base
✓ Team advanced
✓ Checkpoint loaded — 7693/55 games already done, 1 remaining

── Batch 1/1 (1 games) ──


2026-05-07 19:38:52,126 [WARNING] Skipped game 0042500202 after retries: 'NoneType' object has no attribute 'get'



  ✗ 1 games failed across all batches: ['0042500202']
✓ START_POSITION ready (201864 rows)


In [19]:
df = logs.build().get_df()
pos = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_positions/nba_2026_players.csv').rename(
    columns={'name_s26': 'PLAYER_NAME', 'pos': 'POS', 'age': 'AGE'}
)
starting_positions = pd.read_csv('tracking_checkpoint.csv')
df = df.merge(pos, on='PLAYER_NAME', how='left')
df['IS_PLAYOFF'] = 1
df.head()

✓ Built — shape: (1207, 173)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_positions/nba_2026_players.csv'

In [4]:
_s26_path = '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S26.csv'
_s26 = pd.read_csv(_s26_path)

import re
import unicodedata

_suffix_re = re.compile(r"\b(jr|sr|ii|iii|iv|v)\b\.?$", re.I)

def _norm_player_name(name: str) -> str:
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(ch for ch in name if not unicodedata.combining(ch))
    name = name.replace("'", "")
    name = re.sub(r"[^A-Za-z0-9 ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip().lower()
    name = _suffix_re.sub('', name).strip()
    return re.sub(r"\s+", " ", name).strip()

_s26_name_map = {}
for _n in _s26['PLAYER_NAME'].dropna().astype(str).unique():
    _s26_name_map.setdefault(_norm_player_name(_n), set()).add(_n)

if 'PLAYER_NAME' in df.columns:
    df = df.copy()
    df['PLAYER_NAME'] = [
        sorted(_s26_name_map.get(_norm_player_name(n), {str(n)}))[0]
        for n in df['PLAYER_NAME'].astype(str)
    ]
df['STARTING'] = df['START_POSITION'].notna().astype(int)
df['PTS_PER_MIN'] = df['PTS'] / df['MIN'].replace(0,np.nan)
df['AST_PER_MIN'] = df['AST'] / df['MIN'].replace(0,np.nan)
df['REB_PER_MIN'] = df['REB'] / df['MIN'].replace(0,np.nan)
df['IS_HOME'] = df['MATCHUP'].str.contains('vs', na=False).astype(int)
le = LabelEncoder()
df['POSITION_ENCODED'] = le.fit_transform(df['POS'])
df.to_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P26.csv', index=False)

# quick sanity check
if 'PLAYER_NAME' in df.columns:
    missing = sorted(set(df['PLAYER_NAME'].astype(str)) - set(_s26['PLAYER_NAME'].astype(str)))
    print(f"P26 unique names: {df['PLAYER_NAME'].nunique()} | missing vs s26: {len(missing)}")
    if missing:
        print('Missing:', missing[:50])


P26 unique names: 230 | missing vs s26: 1
Missing: ['CJ McCollum']


In [ ]:
"""
BettingPros NBA Prop Bets Scraper
Fetches each market separately to get all props.
"""

import requests
import json
import csv
from datetime import date

BASE_URL = "https://api.bettingpros.com/v3/props"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/121.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.bettingpros.com",
    "Referer": "https://www.bettingpros.com/",
}

MARKET_NAMES = {
    156: "Points",
    151: "Assists",
    157: "Rebounds",
    335: "Pts+Ast",
    336: "Pts+Reb",
    337: "Reb+Ast",
    338: "Pts+Reb+Ast",
    152: "Steals",
    160: "Blocks",
    162: "3-Pointers Made",
}


def fetch_market(target_date: str, market_id: int, limit: int = 100, offset: int = 0) -> dict:
    params = {
        "limit": limit,
        "offset": offset,
        "sport": "NBA",
        "market_id": market_id,
        "date": target_date,
        "include_selections": "false",
        "include_filter_graphs": "false",
        "data_points": 8,
        "min_odds": -1000,
        "max_odds": 1000,
        "ev_threshold_min": -0.4,
        "ev_threshold_max": 0.4,
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def parse_props(data: dict) -> list[dict]:
    rows = []
    for prop in data.get("props", []):
        proj = prop.get("projection") or {}
        rows.append({
            "player": prop.get("participant", {}).get("name", "Unknown"),
            "prop":   MARKET_NAMES.get(prop.get("market_id"), prop.get("market_id")),
            "line":   prop.get("over", {}).get("line"),
            "proj":   proj.get("value"),
            "side":   proj.get("recommended_side"),
            "diff":   proj.get("diff"),
        })
    return rows


def scrape_all(target_date: str) -> list[dict]:
    all_rows = []

    print(f"Fetching props for {target_date}...")
    for market_id, market_name in MARKET_NAMES.items():
        seen   = set()
        offset = 0

        while True:
            data = fetch_market(target_date, market_id, limit=500, offset=offset)
            rows = parse_props(data)

            if not rows:
                break

            new_rows = []
            for r in rows:
                key = (r["player"], r["prop"], r["line"])
                if key not in seen:
                    seen.add(key)
                    new_rows.append(r)

            if not new_rows:
                break

            all_rows.extend(new_rows)
            offset += 100

        print(f"  {market_name:<20} → {len([r for r in all_rows if r['prop'] == market_name])} props")

    print(f"\n  Total: {len(all_rows)} props")
    return all_rows


def save_csv(rows: list[dict], filename: str):
    if not rows:
        print("No data to save.")
        return
    fieldnames = list(rows[0].keys())
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} rows → {filename}")


def save_json(rows: list[dict], filename: str):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    print(f"Saved {len(rows)} rows → {filename}")


from datetime import date, timedelta

START_DATE    = date(2025, 4, 1)
END_DATE      = date(2025, 4, 15)
OUTPUT_FORMAT = "csv"
OUTPUT_NAME   = "nba_props"

current = START_DATE
while current <= END_DATE:
    target = str(current)
    print(f"\n{'='*50}\nProcessing {target}\n{'='*50}")
    
    rows = scrape_all(target)
    
    if rows:
        if OUTPUT_FORMAT in ("csv", "both"):
            save_csv(rows, f"historical_odds/{target}.csv")
        if OUTPUT_FORMAT in ("json", "both"):
            save_json(rows, f"historical_odds/{target}.json")
    else:
        print(f"  No props found for {target}, skipping.")
    
    current += timedelta(days=1)

print("\nDone!")